In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/drug-information/formatted_EMDEX.txt
/kaggle/input/drug-information/formatted_PTHB9.txt
/kaggle/input/drug-information/formatted_BNF.txt


In [23]:
!pip install langchain chromadb sentence-transformers tqdm PyMuPDF langchain-community

In [24]:
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from tqdm import tqdm

# Load text files
def load_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

# Load BNF & EMDEX
bnf_text = load_text("/kaggle/input/drug-information/formatted_BNF.txt")
emdex_text = load_text("/kaggle/input/drug-information/formatted_EMDEX.txt")
pthb9_text = load_text("/kaggle/input/drug-information/formatted_PTHB9.txt")

# Text splitter for better searchability
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)

# Split each document into chunks
bnf_chunks = text_splitter.split_text(bnf_text)
emdex_chunks = text_splitter.split_text(emdex_text)
pthb9_chunks = text_splitter.split_text(pthb9_text)

print(f"🔹 BNF Chunks: {len(bnf_chunks)} | 🔹 EMDEX Chunks: {len(emdex_chunks)} | 🔹 PTHB9 Chunks: {len(pthb9_chunks)}")

🔹 BNF Chunks: 11939 | 🔹 EMDEX Chunks: 3009 | 🔹 PTHB9 Chunks: 2536


In [ ]:
# Initialize ChromaDB storage
chroma_db_path = "./chroma_db"
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

# Store separately for BNF-84
bnf_docs = [Document(page_content=chunk, metadata={"source": "BNF-84"}) for chunk in bnf_chunks]

# Store separately for EMDEX
emdex_docs = [Document(page_content=chunk, metadata={"source": "EMDEX"}) for chunk in emdex_chunks]

# Store separately for PTHB9
pthb9_docs = [Document(page_content=chunk, metadata={"source": "PTHB9"}) for chunk in pthb9_chunks]

# Create vector store with metadata support
vectorstore = Chroma.from_documents(
    documents=bnf_docs + emdex_docs + pthb9_docs,  # Include all datasets
    embedding=embedding_model,
    persist_directory=chroma_db_path)

print("✅ BNF-84, EMDEX, and PTHB9 Data Stored in ChromaDB!")

### Download the Chroma DB

In [9]:
import shutil

# Zip the ChromaDB directory
shutil.make_archive("chroma_db_backup", "zip", "./chroma_db")

'/kaggle/working/chroma_db_backup.zip'

In [25]:
# Load the same embedding model used during storage
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

# Load the ChromaDB database
chroma_db_path = "./chroma_db"
vectorstore = Chroma(persist_directory=chroma_db_path, embedding_function=embedding_model)

print("✅ ChromaDB Loaded Successfully!")

✅ ChromaDB Loaded Successfully!


In [28]:
def retrieve_drug_info(query, top_k=3):
    """Retrieve top 3 results from each dataset in ChromaDB."""
    
    # Query separately for each dataset
    results = []
    for source in ["BNF-84", "EMDEX", "PTHB9"]:
        source_results = vectorstore.similarity_search(
            query, k=top_k, filter={"source": source}  # Filter by dataset
        )
        results.extend(source_results)  # Merge results

    # Format retrieved data
    retrieved_texts = {source: [] for source in ["BNF-84", "EMDEX", "PTHB9"]}
    for res in results:
        retrieved_texts[res.metadata["source"]].append(res.page_content)

    # Summarize retrieved content for LLM input
    summarized_info = "\n\n".join([
    f"Source: {src}\n" + "\n".join(retrieved_texts[src]) for src in retrieved_texts if retrieved_texts[src]])


    return summarized_info

### Test with Similarity Search

In [31]:
# Example search query
search_query = "What is the dosage for paracetamol?"
similarity_result = retrieve_drug_info(search_query)
similarity_result

'Source: BNF-84\nmg every 4–6 hours; maximum 4 doses per day ▶Child 8–9 years: 360–375 mg every 4–6 hours; maximum 4 doses per day ▶Child 10–11 years: 480–500 mg every 4–6 hours; maximum 4 doses per day ▶Child 12–15 years: 480–750 mg every 4–6 hours; maximum 4 doses per day ▶Child 16–17 years: 0.5–1 g every 4–6 hours; maximum 4 doses per day ▶BY RECTUM ▶Child 3–11 months: 60–125 mg every 4–6 hours as required; maximum 4 doses per day ▶Child 1–4 years: 125–250 mg every 4–6 hours as required; maximum 4 doses per day ▶Child 5–11 years: 250–500 mg every 4–6 hours as required; maximum 4 doses per day ▶Child 12–17 years: 500 mg every 4–6 hours Post-immunisation pyrexia in infants ▶BY MOUTH ▶Child 2–3 months: 60 mg for 1 dose, then 60 mg after 4–6 hours if required ▶Child 4 months: 60 mg for 1 dose, then 60 mg after 4–6 hours; maximum 4 doses per day Acute migraine ▶BY MOUTH ▶Adult: 1 g for 1 dose, to be taken as soon as migraine symptoms develop l UNLICENSED USE ▶In children Paracetamol oral